# Mining with MeerKAT EDR in the DP2 coadd footprint

Selects the MeerKAT Mining-with-MeerKAT EDR images and L-band source catalog entries that fall within the Rubin DP2 footprint.

Here the footprint is where DP2 **`deep_coadd` images actually exist**, found from the RSP ObsCore table (`ivoa.ObsCore`, `obs_collection = 'LSST.DP2'`). Each coadd patch has an outline (`s_region`), and the patch outlines for each band are rasterised onto a HEALPix map. The footprint is the sky where every band in `REQUIRED_BANDS` has a coadd. The result is stored as a MOC (Multi-Order Coverage map).

This differs from `Mining-with-MeerKAT_DP2Footprint.ipynb`, which builds the footprint from the DP2 `Visit` table. Some areas with DP2 visits have no DP2 coadds (e.g. the XMM-LSS field around RA 37°, Dec −4°). ObsCore has no depth information for coadds (exposure time is empty), so this footprint can't be cut on the number of visits.

The two MeerKAT catalogs are read directly from their URLs. They are cached by astropy (in `~/.astropy/cache`) after the first download (the source catalog is ~160 MB). Use `clear_download_cache()` to force a fresh download.

In [1]:
# mocpy may not be in the RSP environment; install it into your user area if needed.
# After a fresh install, restart the kernel (Kernel > Restart Kernel) and run the notebook again.
try:
    import mocpy
except ImportError:
    %pip install --user --quiet mocpy

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


In [2]:
import os
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import astropy.table as atpy
import astropy.units as u
from astropy.coordinates import Angle, Longitude, Latitude, SkyCoord
from mocpy import MOC
from astropy.wcs import WCS
from astropy.visualization.wcsaxes.frame import EllipticalFrame
from astropy.utils.data import download_file, clear_download_cache
from lsst.rsp import get_tap_service

In [3]:
DATA_DIR = os.getcwd()   # outputs (and the coadd patch cache) go next to this notebook
NSIDE = 1024             # HEALPix resolution for the coverage maps (~3.4 arcmin pixels; a patch is ~11 arcmin)

# Footprint: sky where every band listed has a DP2 deep_coadd
REQUIRED_BANDS = ["g", "r", "z"]

os.makedirs(DATA_DIR, exist_ok=True)
PATCH_CACHE = os.path.join(DATA_DIR, "DP2_coadd_patches.fits")
OUT_PREFIX = "InDP2Coadd_"   # output catalog prefix (differs from the Visit-based notebook's "InDP2_")

## Query DP2 deep_coadd patches from ObsCore

ObsCore has one row per patch per band (~925,000 rows). To keep memory use down, the patch outlines (`s_region`, the same in every band) are fetched once per patch with `DISTINCT`, and the band list is fetched separately without them. The first run takes about two minutes. The result is reduced to one row per patch (outline corners plus a has-coadd flag per band) and cached in `DP2_coadd_patches.fits`. Delete the cache to re-query.

In [4]:
ALL_BANDS = ["u", "g", "r", "i", "z", "y"]

def runQuery(service, query):
    """Run an async TAP query and return the result as an astropy table."""
    job = service.submit_job(query)
    job.run()
    job.wait(phases=["COMPLETED", "ERROR"])
    job.raise_if_error()
    tab = job.fetch_result().to_table()
    job.delete()
    return tab

if os.path.exists(PATCH_CACHE):
    patchTab = atpy.Table.read(PATCH_CACHE)
else:
    service = get_tap_service("tap")
    where = "FROM ivoa.ObsCore WHERE obs_collection = 'LSST.DP2' AND dataproduct_subtype = 'lsst.deep_coadd'"
    polyTab = runQuery(service, f"SELECT DISTINCT lsst_tract, lsst_patch, s_region {where}")
    bandTab = runQuery(service, f"SELECT lsst_band, lsst_tract, lsst_patch {where}")

    # One row per (tract, patch)
    patchKey = lambda t: np.array(t["lsst_tract"], dtype=np.int64) * 100000 + np.array(t["lsst_patch"], dtype=np.int64)
    polyKey = patchKey(polyTab)
    uKey, first = np.unique(polyKey, return_index=True)
    patchTab = atpy.Table()
    patchTab["tract"] = np.array(polyTab["lsst_tract"][first], dtype=np.int32)
    patchTab["patch"] = np.array(polyTab["lsst_patch"][first], dtype=np.int32)
    # s_region is "POLYGON ICRS ra1 dec1 ra2 dec2 ra3 dec3 ra4 dec4"
    corners = np.array([str(s).split()[2:] for s in polyTab["s_region"][first]], dtype=float).reshape(-1, 4, 2)
    patchTab["ra"] = corners[..., 0]
    patchTab["dec"] = corners[..., 1]
    del polyTab

    rowPatch = np.searchsorted(uKey, patchKey(bandTab))   # patch index of each (band, patch) row
    bandArr = np.array(bandTab["lsst_band"], dtype=str)
    for b in ALL_BANDS:
        has = np.zeros(len(uKey), dtype=bool)
        has[rowPatch[bandArr == b]] = True
        patchTab[f"has_{b}"] = has
    del bandTab
    patchTab.write(PATCH_CACHE, overwrite=True)

print(f"{len(patchTab)} DP2 coadd patches")
print("Patches with a coadd, per band:", {b: int(patchTab[f"has_{b}"].sum()) for b in ALL_BANDS})

197105 DP2 coadd patches
Patches with a coadd, per band: {'u': 114892, 'g': 155115, 'r': 171856, 'i': 177588, 'z': 166412, 'y': 139597}


## Per-band coverage maps

Each patch outline is rasterised onto the HEALPix grid (pixels whose centres fall inside the patch). A patch counts as covered in a band if a `deep_coadd` exists for it, even if it is only partly filled with data (e.g. at the survey edge).

In [5]:
npix = hp.nside2npix(NSIDE)
pixArea = hp.nside2pixarea(NSIDE, degrees=True)

# Pixels inside each patch (hp.ang2vec needs 1-D input, so flatten and reshape)
cornerVecs = hp.ang2vec(np.ravel(patchTab["ra"]), np.ravel(patchTab["dec"]), lonlat=True).reshape(-1, 4, 3)
patchPix = [hp.query_polygon(NSIDE, v, nest=True) for v in cornerVecs]
allPix = np.concatenate(patchPix)
pixPatch = np.repeat(np.arange(len(patchPix)), [len(p) for p in patchPix])   # patch index of each entry in allPix

coverageMaps = {}
for b in ALL_BANDS:
    m = np.zeros(npix, dtype=bool)
    m[allPix[np.array(patchTab[f"has_{b}"])[pixPatch]]] = True
    coverageMaps[b] = m
    print(f"{b}: {m.sum() * pixArea:6.0f} deg^2")

u:   2535 deg^2
g:   3427 deg^2
r:   3798 deg^2
i:   3932 deg^2
z:   3678 deg^2


y:   3085 deg^2


## Footprint

Change `REQUIRED_BANDS` in the config cell (or call `footprintMOC()` directly) to change the definition of usable sky. Re-running this cell is quick, because the coverage maps are already built.

In [6]:
def mocFromMask(mask):
    """MOC made from the HEALPix pixels (nested, NSIDE) where mask is True."""
    ipix = np.where(mask)[0].astype(np.uint64)
    order = hp.nside2order(NSIDE)
    return MOC.from_healpix_cells(ipix, np.full(len(ipix), order, dtype=np.uint8), max_depth=order)

def footprintMOC(requiredBands):
    """MOC of the sky where every band in requiredBands has a DP2 deep_coadd."""
    mask = np.ones(npix, dtype=bool)
    for b in requiredBands:
        if b not in coverageMaps:
            raise ValueError(f"Unknown band {b!r}; available bands: {list(coverageMaps)}")
        mask &= coverageMaps[b]
    return mocFromMask(mask)

mocAny = mocFromMask(np.any(list(coverageMaps.values()), axis=0))   # coadd in any band, for comparison
moc = footprintMOC(REQUIRED_BANDS)

sqDeg = 4 * np.pi * (180 / np.pi)**2
print(f"DP2 coadds in any band: {mocAny.sky_fraction * sqDeg:.0f} deg^2")
print(f"Footprint (coadds in {'+'.join(REQUIRED_BANDS)}): {moc.sky_fraction * sqDeg:.0f} deg^2")
moc.save(os.path.join(DATA_DIR, "DP2_coadd_footprint_moc.fits"), format="fits", overwrite=True)

DP2 coadds in any band: 4363 deg^2
Footprint (coadds in g+r+z): 3068 deg^2


## Select MeerKAT EDR images and sources inside the footprint

In [7]:
EDR_URL = "https://extragalactic.phys.wits.ac.za/mining-with-meerkat/EDR/data/"

def getTabInDP2(url, RACol, decCol):
    """Download (or read from the astropy cache) the catalog at url, and return the rows within the DP2 footprint."""
    tab = atpy.Table.read(download_file(url, cache=True, show_progress=False))
    hitMask = moc.contains_lonlat(Angle(tab[RACol], unit="deg"), Angle(tab[decCol], unit="deg"))
    print(f"{os.path.basename(url)}: {hitMask.sum()} / {len(tab)} in DP2")
    return tab[hitMask]

In [8]:
imagesTab = getTabInDP2(EDR_URL + "images_EDR.fits", RACol="centre_RADeg", decCol="centre_decDeg")
imagesTab.write(os.path.join(DATA_DIR, OUT_PREFIX + "images_EDR.fits"), overwrite=True)

sourcesTab = getTabInDP2(EDR_URL + "survey_catalog_L_EDR.fits", RACol="RA", decCol="DEC")
sourcesTab.write(os.path.join(DATA_DIR, OUT_PREFIX + "survey_catalog_L_EDR.fits"), overwrite=True)

images_EDR.fits: 10 / 141 in DP2


survey_catalog_L_EDR.fits: 15461 / 296942 in DP2


## Footprint map with MeerKAT EDR image positions

In [9]:
allImagesTab = atpy.Table.read(download_file(EDR_URL + "images_EDR.fits", cache=True, show_progress=False))

# Full-sky Aitoff WCS whose pixel grid just encloses the projected sky (4*sqrt(2) x 2*sqrt(2) radians)
cdelt = 0.1  # deg per pixel
nx, ny = int(np.ceil(4 * np.sqrt(2) * 180 / np.pi / cdelt)), int(np.ceil(2 * np.sqrt(2) * 180 / np.pi / cdelt))
wcs = WCS(naxis=2)
wcs.wcs.ctype = ["RA---AIT", "DEC--AIT"]
wcs.wcs.crval = [0, 0]
wcs.wcs.crpix = [nx / 2 + 0.5, ny / 2 + 0.5]
wcs.wcs.cdelt = [-cdelt, cdelt]
wcs.pixel_shape = (nx, ny)

fig = plt.figure(figsize=(14, 7.5))
ax = fig.add_axes([0.03, 0.09, 0.94, 0.84], projection=wcs, frame_class=EllipticalFrame)
ax.set_xlim(-0.5, nx - 0.5); ax.set_ylim(-0.5, ny - 0.5)
mocAny.fill(ax=ax, wcs=wcs, alpha=0.3, fill=True, facecolor="lightgrey", edgecolor="none", label="DP2 coadds, any band")
moc.fill(ax=ax, wcs=wcs, alpha=0.6, fill=True, facecolor="tab:blue", edgecolor="none", label="DP2 footprint after cut")
moc.border(ax=ax, wcs=wcs, color="tab:blue")
world = ax.get_transform("world")
ax.scatter(allImagesTab["centre_RADeg"], allImagesTab["centre_decDeg"], transform=world,
           s=14, facecolors="none", edgecolors="black", linewidths=0.8,
           label=f"MeerKAT EDR images ({len(allImagesTab)})")
ax.scatter(imagesTab["centre_RADeg"], imagesTab["centre_decDeg"], transform=world,
           s=20, color="tab:red", label=f"inside footprint ({len(imagesTab)})")
ax.coords.grid(True, color="grey", alpha=0.4, linestyle=":")
ax.coords[0].set_format_unit(u.deg); ax.coords[0].set_major_formatter("d")
ax.coords[1].set_major_formatter("d")
ax.coords[0].set_axislabel(""); ax.coords[1].set_axislabel("")
ax.set_title(f"DP2 coadd footprint ({'+'.join(REQUIRED_BANDS)}) and MeerKAT EDR images (RA, Dec in deg)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=4, frameon=False)
fig.savefig(os.path.join(DATA_DIR, "DP2_coadd_footprint_MeerKAT_EDR.png"), dpi=150, bbox_inches="tight")
plt.show()

/tmp/ipykernel_705964/2464684711.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
